# Quant Risk Core: Advanced Risk Simulations
This notebook provides interactive simulations for Market and Credit Risk, utilizing the `quant_risk_core` engine.

## 1. GARCH(1,1) Volatility Forecasting Simulation
We simulate the out-of-sample volatility decay/persistence across a 30-day horizon.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from market_risk.volatility import GARCHEngine

np.random.seed(42)
# Synthetic returns with a volatility spike
base_vol = 0.01
returns = np.random.normal(0, base_vol, 500)
returns[-50:] *= 4  # Recent spike
returns_series = pd.Series(returns)

garch = GARCHEngine(p=1, q=1)
garch.fit(returns_series)
forecast_vol = garch.forecast_volatility(horizon=30)
in_sample_vol = garch.conditional_volatility()

fig = go.Figure()
fig.add_trace(go.Scatter(y=in_sample_vol, name='In-Sample Volatility'))
x_forecast = np.arange(len(in_sample_vol), len(in_sample_vol) + 30)
fig.add_trace(go.Scatter(x=x_forecast, y=forecast_vol, name='30D Forecast', line=dict(dash='dash', color='red')))
fig.update_layout(title='GARCH(1,1) Volatility: In-Sample vs 30-Day Forecast', xaxis_title='Days', yaxis_title='Volatility')
fig.show()

## 2. Monte Carlo VaR Simulation: Path Convergence
Visualizing 100 sample paths of Geometric Brownian Motion and the resulting distribution of 50,000 terminal returns.

In [2]:
from market_risk.estimators import RiskEngine, simulate_paths

S0, mu, sigma, horizon = 100, 0.05, 0.2, 21 # 1 month
paths_to_plot = 100
total_paths = 50000

# Simulate paths for visualization
sample_paths = np.zeros((paths_to_plot, horizon + 1))
sample_paths[:, 0] = S0
for i in range(paths_to_plot):
    for t in range(horizon):
        z = np.random.standard_normal()
        sample_paths[i, t+1] = sample_paths[i, t] * np.exp((mu/252 - 0.5*(sigma/np.sqrt(252))**2) + (sigma/np.sqrt(252))*z)

fig_paths = go.Figure()
for i in range(paths_to_plot):
    fig_paths.add_trace(go.Scatter(y=sample_paths[i], mode='lines', line=dict(width=1), opacity=0.3, showlegend=False))
fig_paths.update_layout(title=f'Monte Carlo: {paths_to_plot} Sample GBM Paths', xaxis_title='Days', yaxis_title='Price')
fig_paths.show()

engine = RiskEngine(confidence_levels=[0.99])
mc_results = engine.monte_carlo_var_es(S0, mu/252, sigma/np.sqrt(252), horizon, paths=total_paths)
print(f'Monte Carlo 99% VaR (1M): {mc_results["VaR_0.99"]:.4%}')
print(f'Monte Carlo 99% ES (1M) : {mc_results["ES_0.99"]:.4%}')

Monte Carlo 99% VaR (1M): 12.4002%
Monte Carlo 99% ES (1M) : 13.9730%


## 3. Rolling VaR Backtest Simulation
Simulating a 250-day rolling window backtest. We compare daily returns against the GARCH-predicted 99% VaR threshold.

In [3]:
from market_risk.backtesting import RiskBacktester

np.random.seed(1)
n_days = 500
test_returns = np.random.normal(0, 0.015, n_days)
test_returns[300:350] *= 2.5 # Stress period
test_returns_series = pd.Series(test_returns)

# Fit GARCH once for simplicity in this demo, usually done rolling
garch_bt = GARCHEngine()
garch_bt.fit(test_returns_series)
vols = garch_bt.conditional_volatility()

# Compute Parametric 99% VaR thresholds
risk_eng = RiskEngine(confidence_levels=[0.99])
vars_99 = [risk_eng.parametric_var_es(0, v, 'normal')['VaR_0.99'] for v in vols]

fig_bt = go.Figure()
fig_bt.add_trace(go.Scatter(y=test_returns, name='Returns', mode='markers+lines', marker=dict(size=4)))
fig_bt.add_trace(go.Scatter(y=-np.array(vars_99), name='99% VaR Threshold', line=dict(color='orange')))

# Highlight exceptions
exceptions = np.where(test_returns < -np.array(vars_99))[0]
fig_bt.add_trace(go.Scatter(x=exceptions, y=test_returns[exceptions], mode='markers', 
                         marker=dict(color='red', size=8, symbol='x'), name='Exceptions'))

fig_bt.update_layout(title='Backtesting Simulation: Daily Returns vs GARCH VaR', xaxis_title='Days', yaxis_title='Return')
fig_bt.show()

bt_eval = RiskBacktester(0.99).evaluate(test_returns_series, pd.Series(vars_99))
print(f'Basel Zone: {bt_eval["Basel_Zone"]}')
print(f'Exceptions: {bt_eval["Exceptions"]}')

Basel Zone: Green
Exceptions: 4


## 4. Counterparty Credit Exposure Simulation
Simulating the exposure profile for a derivative portfolio. We show the evolution of Expected Exposure (EE) and Potential Future Exposure (PFE).

In [4]:
from credit_risk.counterparty import CounterpartyRiskEngine

time_grid = np.linspace(0, 3, 30) # 3 years
n_paths = 2000
portfolio_paths = np.zeros((n_paths, len(time_grid)))

# Simulate Mean Reverting Portfolio Value (Ornstein-Uhlenbeck style for MTM)
for i in range(n_paths):
    v = 0
    for t in range(len(time_grid)):
        v = v * 0.95 + np.random.normal(0, 5)
        portfolio_paths[i, t] = v

cpty_eng = CounterpartyRiskEngine(time_grid)
cpty_eng.set_portfolio_paths(portfolio_paths)
profiles = cpty_eng.calculate_exposure_profiles(quantile=0.95)

fig_cpty = go.Figure()
fig_cpty.add_trace(go.Scatter(x=time_grid, y=profiles['EE'], name='Expected Exposure (EE)', fill='tozeroy'))
fig_cpty.add_trace(go.Scatter(x=time_grid, y=profiles['PFE'], name='PFE (95%)', line=dict(dash='dot')))
fig_cpty.update_layout(title='Counterparty Exposure Simulation', xaxis_title='Years', yaxis_title='Exposure Value')
fig_cpty.show()